In [1]:
#| default_exp frida

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [4]:
#| export
from os import getenv
model_path = getenv("MODEL")
from rest.gen import generate


In [5]:
model_path = 'fred'

In [6]:
#| export
from optimum.onnxruntime import ORTModelForSeq2SeqLM

In [7]:
#| export
seq_length = 1024

full_path = f'./models/{model_path}'
import torch
from transformers import GPT2Tokenizer, T5ForConditionalGeneration, AutoTokenizer

In [8]:
def convert_and_save_model(model_path, save_dir):
    model = ORTModelForSeq2SeqLM.from_pretrained(model_path, export=True)
    model.save_pretrained(save_dir)
    
    # Also save the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    tokenizer.save_pretrained(save_dir)
    
    print(f"Model and tokenizer saved to {save_dir}")

In [9]:
#convert_and_save_model(full_path, full_path+'/optimized')

Framework not specified. Using pt to export the model.
Using the export variant default. Available variants are:
    - default: The default ONNX variant.

***** Exporting submodel 1/3: T5Stack *****
Using framework PyTorch: 2.5.0a0+b465a5843b.nv24.09
Overriding 1 configuration item(s)
	- use_cache -> False
Saving external data to one file...

***** Exporting submodel 2/3: T5ForConditionalGeneration *****
Using framework PyTorch: 2.5.0a0+b465a5843b.nv24.09
Overriding 1 configuration item(s)
	- use_cache -> True
/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:1092: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if causal_mask.shape[1] < attention_mask.shape[1]:
Saving external data to one file...

***** Exporting submodel 3/3: T5ForConditionalGenera

Model and tokenizer saved to ./models/fred/optimized


In [10]:
#| export
tokenizer = GPT2Tokenizer.from_pretrained(full_path+'/optimized/', eos_token='</s>')
model = ORTModelForSeq2SeqLM.from_pretrained(full_path+'/optimized/', provider="CUDAExecutionProvider")

2024-09-29 16:49:36.751840932 [W:onnxruntime:, session_state.cc:1166 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2024-09-29 16:49:36.751858912 [W:onnxruntime:, session_state.cc:1168 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
2024-09-29 16:49:37.469447409 [W:onnxruntime:, session_state.cc:1166 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2024-09-29 16:49:37.469466409 [W:onnxruntime:, session_state.cc:1168 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
2024-09-29 16:49:38.253896086 [W:onnxrun

In [11]:
#| export
from front.common import process_seq

def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool, temperature:float=0.5):
    lm_text = '<LM>' + prompt
    input_ids=torch.tensor([tokenizer.encode(lm_text)]).cuda()
    output_ids = model.generate(input_ids, do_sample=True, temperature=temperature, repetition_penalty=5.0, typical_p=0.9, top_k=10, top_p=0.95, #watermark=False,
                        max_new_tokens=length, 
                        num_return_sequences=num_samples,)

    result = [tokenizer.decode(o[1:]).replace('\n', ' ') for o in output_ids]
    result = process_seq(result)
    return result


In [12]:
%%time
get_sample('<LM>На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 729 ms, sys: 63 ms, total: 792 ms
Wall time: 824 ms


[' – просто говно. –\xa0 А ты кто? Ты сам-то себя слышишь, когда говоришь такое про других людей?! Я тебе не какой‑ нибудь там «Лев Толстой»! У меня есть имя и фамилия!',
 ' – просто мудак. –\xa0 А ты кто? Ты тоже Лев Толстой, только в юбке и с косой?',
 ' — просто свинья».  «А что, я не Лев Толстой?»— спросил он.',
 ' — просто говно».  В ответ я сказал: «А что, это правда?» Он засмеялся.']

In [13]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 576 ms, sys: 1.67 ms, total: 577 ms
Wall time: 570 ms


[' – просто говно. –\xa0 А на деле ты кто?      Я, конечно же… Но это уже совсем другая история!',
 ' – говно. –\xa0 А что, я и есть Лев Толстой? Ну-ка скажи! Ты же сам сказал: «Я не знаю Льва Толстого». Вот ты его знаешь… Как он выглядит на самом деле?',
 ' — говно. —\xa0 А ты, значит… Ты тоже? Ну-ну! Я не про то спрашиваю: «А что это за книга?» Или я ошибаюсь и тебе нравится эта книжка только потому?.. Да ладно уж!..',
 ' – говно». –\xa0 А я не Лев Толстой. Я просто хороший писатель, который пишет про хороших людей хорошие книги… И вообще это все неправда! Вот вы говорите «хорошо», а сами что делаете?']

In [14]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 591 ms, sys: 659 μs, total: 591 ms
Wall time: 584 ms


[' – просто говно. –\xa0 А на деле я не Лев Толстой, а Николай Второй… И вообще ты мне надоел со своими шуточками! Я тебе что сказал? Ты меня понял?!',
 ' – говно. –\xa0 Это не я, это ты так думаешь… А на самом деле все наоборот: говном был Лев Толстой! И сейчас он тоже в дерьме по уши». Вот как надо говорить с людьми!',
 ' – просто говно. –\xa0 А ты, значит…? Ну и ну! Как же это тебя угораздило-то так попасться на удочку к этому гаду?! Ты хоть знаешь его имя?..',
 ' – просто говно».  А потом, когда я уже был на пути к цели и в голове у меня все смешалось от счастья (а может быть это была боль), вдруг пришло понимание того простого факта что «Лев Толстой» — не мое имя.']